# LoRA + SFT on DGX Spark: Can We Fine-Tune Granite on a Desktop Supercomputer?

**Algorithm:** LoRA (Low-Rank Adaptation) with Supervised Fine-Tuning  
**Model:** `ibm-granite/granite-3.3-2b-instruct`  
**Hardware:** NVIDIA DGX Spark (GB10, 130.7 GB unified memory)  
**Dataset:** `b-mc2/sql-create-context` (natural language → SQL)  

---

This notebook documents our attempt to run LoRA+SFT fine-tuning on a DGX Spark using
[Training Hub](https://github.com/instructlab/training-hub). The DGX Spark is NVIDIA's
"desktop AI supercomputer" — a Grace Blackwell system with 130.7 GB of shared CPU/GPU memory
and an ARM (aarch64) processor. It's a fascinating piece of hardware, but it's also bleeding-edge:
compute capability 12.1, CUDA 13.0, and an ARM CPU mean that many ML libraries don't have
pre-built wheels yet.

We're not just showing the happy path — we're documenting what worked, what broke, and what
workarounds were needed. That's the real story.

## 1. DGX Spark System Profile

Let's see exactly what hardware and software we're working with.

> **Estimated run time:** ~5 seconds

In [1]:
import platform
import subprocess
import torch

print("=" * 60)
print("DGX Spark System Profile")
print("=" * 60)

# CPU & OS
print(f"\nPlatform:    {platform.machine()}")
print(f"Processor:   {platform.processor() or 'ARM (Grace)'}")
print(f"OS:          {platform.platform()}")

# GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    cc = torch.cuda.get_device_capability(0)
    print(f"\nGPU:         {gpu_name}")
    print(f"GPU Memory:  {gpu_mem:.1f} GB")
    print(f"Compute Cap: {cc[0]}.{cc[1]}")
    print(f"CUDA:        {torch.version.cuda}")
else:
    print("\nNo CUDA GPU detected!")

# PyTorch
print(f"\nPyTorch:     {torch.__version__}")
print(f"BF16:        {torch.cuda.is_bf16_supported()}")

# Shared memory (DGX Spark uses unified memory)
import os
total_ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
print(f"\nSystem RAM:  {total_ram:.1f} GB (shared with GPU on DGX Spark)")

DGX Spark System Profile

Platform:    aarch64
Processor:   aarch64
OS:          Linux-6.17.0-1008-nvidia-aarch64-with-glibc2.39

GPU:         NVIDIA GB10
GPU Memory:  121.7 GB
Compute Cap: 12.1
CUDA:        13.0

PyTorch:     2.10.0+cu130
BF16:        True

System RAM:  121.7 GB (shared with GPU on DGX Spark)


/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


## 2. Environment Check — What's Installed, What's Missing

Training Hub's LoRA backend defaults to **Unsloth** for optimized LoRA training. Let's check
what we have and what we're missing.

> **Estimated run time:** ~2 seconds

In [2]:
# Check installed packages relevant to LoRA training
packages = [
    "training-hub", "instructlab-training", "mini-trainer",
    "peft", "bitsandbytes", "trl", "accelerate",
    "transformers", "datasets", "torch",
    "unsloth", "xformers", "flash-attn", "deepspeed",
    "liger-kernel", "mamba-ssm"
]

import importlib.metadata

print(f"{'Package':<25} {'Status':<15} {'Version'}")
print("-" * 55)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:<25} {'INSTALLED':<15} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:<25} {'MISSING':<15} —")

Package                   Status          Version
-------------------------------------------------------
training-hub              INSTALLED       0.5.0
instructlab-training      INSTALLED       0.14.1
mini-trainer              MISSING         —
peft                      INSTALLED       0.18.1
bitsandbytes              INSTALLED       0.48.1
trl                       INSTALLED       0.19.1
accelerate                INSTALLED       1.10.1
transformers              INSTALLED       5.2.0
datasets                  INSTALLED       4.3.0
torch                     INSTALLED       2.10.0+cu130
unsloth                   INSTALLED       2026.3.4
xformers                  INSTALLED       0.0.35
flash-attn                MISSING         —
deepspeed                 INSTALLED       0.18.7
liger-kernel              INSTALLED       0.7.0
mamba-ssm                 MISSING         —


### 2a. Attempting to Install Unsloth

Unsloth is the default backend for `lora_sft()`. Let's try installing it.

> **Expectation:** This will likely fail on DGX Spark. Unsloth has custom CUDA kernels that
> may not have pre-built wheels for ARM aarch64 + compute capability 12.1. We document the
> failure and plan a fallback.

> **Estimated run time:** 30–120 seconds (pip resolve + possible build attempt)

In [3]:
%%time
# Attempt to install unsloth
# This cell documents what happens — success or failure is equally valuable
!pip install unsloth 2>&1 | tail -20

CPU times: user 27.2 ms, sys: 8.64 ms, total: 35.9 ms
Wall time: 2.47 s


In [4]:
# Check if unsloth is now importable
try:
    import unsloth
    print(f"Unsloth installed successfully! Version: {unsloth.__version__}")
    UNSLOTH_AVAILABLE = True
except ImportError as e:
    print(f"Unsloth import failed: {e}")
    print("\nWe'll proceed with a PEFT/TRL fallback approach.")
    UNSLOTH_AVAILABLE = False

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[unsloth.import_fixes|WARNING]Unsloth: Detected broken vLLM binary extension; disabling vLLM imports and continuing import.
Please reinstall via `uv pip install unsloth vllm torchvision torchaudio --torch-backend=auto`.


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth installed successfully! Version: 2026.3.4


In [12]:
%%time
# Also try xformers (used by some LoRA backends for memory-efficient attention)
!pip install xformers 2>&1 | tail -10

try:
    import xformers
    print(f"\nxformers installed! Version: {xformers.__version__}")
except ImportError as e:
    print(f"\nxformers not available: {e}")
    print("Not critical — PyTorch's built-in SDPA attention will be used instead.")


xformers installed! Version: 0.0.35
CPU times: user 13.6 ms, sys: 18.6 ms, total: 32.2 ms
Wall time: 1.53 s


## 3. Memory Estimation — Will It Fit?

Before we try training, let's use Training Hub's memory estimator to check if our
configurations will fit in 130.7 GB of shared memory.

> **Estimated run time:** ~30–60 seconds (downloads model config from HuggingFace on first run)

In [5]:
%%time
from training_hub import estimate

GPU_MEM_BYTES = int(130.7 * 1024**3)  # 130.7 GB in bytes
MODEL = "ibm-granite/granite-3.3-2b-instruct"

print("=" * 60)
print("Memory Estimate: QLoRA (4-bit) — Granite 2B")
print("=" * 60)
qlora_est = estimate(
    training_method="qlora",
    num_gpus=1,
    gpu_memory=GPU_MEM_BYTES,
    model_path=MODEL,
    batch_size=8,
    max_seq_len=512,
    verbose=2,
)

Memory Estimate: QLoRA (4-bit) — Granite 2B
NOTE: Due to its memory efficiency, LoRA's lower bound estimate is lower than the basic sum of the components.
Estimations for ibm-granite/granite-3.3-2b-instruct:


Summary:
The expected amount of memory needed to run this model is about 3.3 GB
The lower and upper bounds are 2.8 - 3.9 GB
If you have 1 GPUs, you will need about 3.3 GB, with bounds of 2.8 - 3.9 GB per GPU


Component Breakdown:
Each GPU will need 1.4 GB to store the model parameters
Each GPU will need 0.2 GB to store the optimizer states
Each GPU will need 0.0 GB to store the gradients
Each GPU will need 0.6 GB to store the intermediate activations
Each GPU will need 0.8 GB to store the outputs
Up to 0.9 GB can be expected as overhead

Decision:
The proposed training setup should work for your hardware.

CPU times: user 53.6 ms, sys: 46.3 ms, total: 99.9 ms
Wall time: 299 ms


In [6]:
%%time
print("=" * 60)
print("Memory Estimate: LoRA (full precision) — Granite 2B")
print("=" * 60)
lora_est = estimate(
    training_method="lora",
    num_gpus=1,
    gpu_memory=GPU_MEM_BYTES,
    model_path=MODEL,
    batch_size=8,
    max_seq_len=512,
    verbose=2,
)

Memory Estimate: LoRA (full precision) — Granite 2B
NOTE: Due to its memory efficiency, LoRA's lower bound estimate is lower than the basic sum of the components.
Estimations for ibm-granite/granite-3.3-2b-instruct:


Summary:
The expected amount of memory needed to run this model is about 7.2 GB
The lower and upper bounds are 6.2 - 7.8 GB
If you have 1 GPUs, you will need about 7.2 GB, with bounds of 6.2 - 7.8 GB per GPU


Component Breakdown:
Each GPU will need 4.9 GB to store the model parameters
Each GPU will need 0.2 GB to store the optimizer states
Each GPU will need 0.0 GB to store the gradients
Each GPU will need 0.6 GB to store the intermediate activations
Each GPU will need 0.8 GB to store the outputs
Up to 1.3 GB can be expected as overhead

Decision:
The proposed training setup should work for your hardware.

CPU times: user 1.36 ms, sys: 67 μs, total: 1.43 ms
Wall time: 1.31 ms


In [7]:
# Summary
def fmt_gb(b):
    return f"{b / 1024**3:.1f} GB"

print("\nMemory Summary:")
print(f"  Available:       {fmt_gb(GPU_MEM_BYTES)}")
print(f"  QLoRA estimate:  {fmt_gb(qlora_est[0])} – {fmt_gb(qlora_est[2])}")
print(f"  LoRA estimate:   {fmt_gb(lora_est[0])} – {fmt_gb(lora_est[2])}")
print(f"\nVerdict: Both should fit comfortably. QLoRA is the safest bet.")


Memory Summary:
  Available:       130.7 GB
  QLoRA estimate:  2.8 GB – 3.9 GB
  LoRA estimate:   6.2 GB – 7.8 GB

Verdict: Both should fit comfortably. QLoRA is the safest bet.


## 4. Dataset Preparation

We're using the **sql-create-context** dataset — it contains natural language questions paired
with SQL queries and table schemas. Small, public, and easy to understand.

Training Hub expects JSONL with a `messages` field containing chat-format conversations.

> **Estimated run time:** ~10–30 seconds (dataset download from HuggingFace)

In [8]:
%%time
from datasets import load_dataset
import json
import os

# Download the dataset
ds = load_dataset("b-mc2/sql-create-context", split="train")
print(f"Dataset size: {len(ds)} examples")
print(f"Columns: {ds.column_names}")
print(f"\nSample entry:")
print(json.dumps(ds[0], indent=2))

Dataset size: 78577 examples
Columns: ['answer', 'question', 'context']

Sample entry:
{
  "answer": "SELECT COUNT(*) FROM head WHERE age > 56",
  "question": "How many heads of the departments are older than 56 ?",
  "context": "CREATE TABLE head (age INTEGER)"
}
CPU times: user 46.3 ms, sys: 19.4 ms, total: 65.7 ms
Wall time: 1.02 s


In [9]:
# Convert to Training Hub's messages JSONL format
# We'll use a subset (2000 examples) to keep training time reasonable

SUBSET_SIZE = 2000
DATA_PATH = "data/sql_train.jsonl"

os.makedirs("data", exist_ok=True)

system_prompt = (
    "You are a SQL expert. Given a natural language question and the relevant table schema, "
    "write the correct SQL query."
)

with open(DATA_PATH, "w") as f:
    for i, row in enumerate(ds):
        if i >= SUBSET_SIZE:
            break
        user_content = f"Question: {row['question']}\n\nContext:\n{row['context']}"
        record = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": row["answer"]},
            ]
        }
        f.write(json.dumps(record) + "\n")

print(f"Wrote {min(SUBSET_SIZE, len(ds))} examples to {DATA_PATH}")
print(f"File size: {os.path.getsize(DATA_PATH) / 1024:.1f} KB")

# Preview first record
with open(DATA_PATH) as f:
    sample = json.loads(f.readline())
print(f"\nPreview:")
print(json.dumps(sample, indent=2))

Wrote 2000 examples to data/sql_train.jsonl
File size: 1025.3 KB

Preview:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a SQL expert. Given a natural language question and the relevant table schema, write the correct SQL query."
    },
    {
      "role": "user",
      "content": "Question: How many heads of the departments are older than 56 ?\n\nContext:\nCREATE TABLE head (age INTEGER)"
    },
    {
      "role": "assistant",
      "content": "SELECT COUNT(*) FROM head WHERE age > 56"
    }
  ]
}


## 5. Training Run — LoRA+SFT with Training Hub

Now for the main event. We'll try Training Hub's `lora_sft()` function.

**Strategy:**
1. If Unsloth is available → use it (default backend)
2. If Unsloth failed → try with `backend="peft"` or manual PEFT/TRL approach

> **Estimated run time:** 5–15 minutes for QLoRA on Granite 2B with 2000 examples
> (includes model download on first run: ~2 min, plus ~3–10 min training)

In [10]:
import time
import os

CKPT_DIR = "checkpoints/lora_sft_granite2b"
os.makedirs(CKPT_DIR, exist_ok=True)

print("Starting LoRA+SFT training...")
print(f"  Model:    {MODEL}")
print(f"  Data:     {DATA_PATH}")
print(f"  Output:   {CKPT_DIR}")
print(f"  Backend:  {'unsloth' if UNSLOTH_AVAILABLE else 'unsloth (may fallback)'}")
print()

Starting LoRA+SFT training...
  Model:    ibm-granite/granite-3.3-2b-instruct
  Data:     data/sql_train.jsonl
  Output:   checkpoints/lora_sft_granite2b
  Backend:  unsloth



In [11]:
from training_hub import lora_sft

start_time = time.time()

try:
    result = lora_sft(
        model_path=MODEL,
        data_path=os.path.abspath(DATA_PATH),
        ckpt_output_dir=os.path.abspath(CKPT_DIR),
        # QLoRA 4-bit quantization — let Training Hub handle the BNB config
        load_in_4bit=True,
        # LoRA config
        lora_r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        # Training config
        num_epochs=1,
        effective_batch_size=8,
        max_seq_len=512,
        learning_rate=2e-4,
        lr_scheduler="cosine",
        warmup_steps=10,
        bf16=True,
        # Single GPU
        nproc_per_node=1,
        # Logging
        logging_steps=25,
        save_steps=500,
    )
    elapsed = time.time() - start_time
    print(f"\nTraining completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

except Exception as e:
    elapsed = time.time() - start_time
    print(f"\nTraining failed after {elapsed:.1f}s")
    print(f"Error: {type(e).__name__}: {e}")
    print("\nSee the Troubleshooting section below for analysis and workarounds.")
    import traceback
    traceback.print_exc()

==((====))==  Unsloth 2026.3.4: Fast Granite patching. Transformers: 5.2.0. vLLM: 0.15.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

ibm-granite/granite-3.3-2b-instruct does not have a padding token! Will use pad_token = <|end_of_text|>.
Unsloth: Making `model.base_model.model.model` require gradients


Generating train split: 0 examples [00:00, ? examples/s]

...

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/2000 [00:00<?, ? examples/s]

df: /home/frank/.triton/autotune: No such file or directory
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status



Training failed after 48.7s
Error: ValueError: None is not supported, only azure_ml, comet_ml, mlflow, neptune, tensorboard, trackio, wandb, codecarbon, clearml, dagshub, flyte, dvclive, swanlab are supported.

See the Troubleshooting section below for analysis and workarounds.


Traceback (most recent call last):
  File "/tmp/ipykernel_181208/1440538111.py", line 6, in <module>
    result = lora_sft(
             ^^^^^^^^^
  File "/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/training_hub/algorithms/lora.py", line 817, in lora_sft
    return algorithm.train(
           ^^^^^^^^^^^^^^^^
  File "/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/training_hub/algorithms/lora.py", line 584, in train
    return self.backend.execute_training(params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/training_hub/algorithms/lora.py", line 61, in execute_training
    trainer = SFTTrainer(
              ^^^^^^^^^^^
  File "/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/unsloth/trainer.py", line 409, in new_init
    original_init(self, *args, **kwargs)
  File "/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/unsloth/trainer.py", line 314, in new_init
    original_init(

## 6. Fallback — Manual PEFT/TRL Approach

If Training Hub's `lora_sft()` failed (likely due to Unsloth unavailability), we can try
a manual LoRA fine-tuning approach using PEFT and TRL directly. These libraries are
pure Python with no custom CUDA kernels, so they should work on any platform.

> **Run this section only if the Training Hub call above failed.**

> **Estimated run time:** 5–15 minutes (model loading ~1–2 min, training ~3–10 min)

In [20]:
# Manual PEFT/TRL fallback
# Uncomment and run this cell if lora_sft() failed above

MANUAL_FALLBACK = False  # Set to True if needed

if MANUAL_FALLBACK:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig
    from datasets import load_dataset
    import torch

    # 4-bit quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    print("Loading model with 4-bit quantization...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model)

    # LoRA config
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Load our JSONL dataset
    train_ds = load_dataset("json", data_files=DATA_PATH, split="train")

    # Format for TRL's SFTTrainer
    def format_messages(example):
        return {"text": tokenizer.apply_chat_template(
            example["messages"], tokenize=False, add_generation_prompt=False
        )}

    train_ds = train_ds.map(format_messages)

    # Training config
    training_args = SFTConfig(
        output_dir=CKPT_DIR + "_manual",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        max_seq_length=512,
        bf16=True,
        logging_steps=25,
        save_steps=500,
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        processing_class=tokenizer,
    )

    print("\nStarting manual PEFT/TRL training...")
    start_time = time.time()
    trainer.train()
    elapsed = time.time() - start_time
    print(f"\nManual training completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

    # Save
    trainer.save_model(CKPT_DIR + "_manual/final")
    print(f"Model saved to {CKPT_DIR}_manual/final")
else:
    print("Manual fallback not needed — Training Hub succeeded (or set MANUAL_FALLBACK=True to try).")

Manual fallback not needed — Training Hub succeeded (or set MANUAL_FALLBACK=True to try).


## 7. GPU Memory Usage

Let's check how much memory training actually used.

In [21]:
if torch.cuda.is_available():
    allocated = torch.cuda.max_memory_allocated() / 1024**3
    reserved = torch.cuda.max_memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3

    print("Peak GPU Memory Usage:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")
    print(f"  Total:     {total:.1f} GB")
    print(f"  Headroom:  {total - reserved:.1f} GB")
else:
    print("No CUDA device available for memory stats.")

Peak GPU Memory Usage:
  Allocated: 0.01 GB
  Reserved:  0.02 GB
  Total:     121.7 GB
  Headroom:  121.7 GB


## 8. Bonus: QLoRA on Granite 8B

The DGX Spark has enough memory that we might be able to QLoRA fine-tune the 8B model too.
The estimator says 7.8–10.6 GB — well within our 130.7 GB budget.

> **This is a stretch goal.** Skip if time is limited.

> **Estimated run time:** 15–45 minutes for 8B QLoRA (model download ~5 min, training ~10–40 min)

In [22]:
%%time
# Memory estimate for 8B QLoRA
MODEL_8B = "ibm-granite/granite-3.3-8b-instruct"

print("Memory Estimate: QLoRA (4-bit) — Granite 8B")
print("=" * 60)
qlora_8b_est = estimate(
    training_method="qlora",
    num_gpus=1,
    gpu_memory=GPU_MEM_BYTES,
    model_path=MODEL_8B,
    batch_size=4,
    max_seq_len=512,
    verbose=2,
)
print(f"\nEstimate: {fmt_gb(qlora_8b_est[0])} – {fmt_gb(qlora_8b_est[2])}")
print(f"Available: {fmt_gb(GPU_MEM_BYTES)}")
print("Verdict: Should fit!")

Memory Estimate: QLoRA (4-bit) — Granite 8B
NOTE: The memory needed for this QLoRA training setup is bounded by pre-quantized size of the model.
You can only reduce the memory further by using a smaller model.
NOTE: Due to its memory efficiency, LoRA's lower bound estimate is lower than the basic sum of the components.
Estimations for ibm-granite/granite-3.3-8b-instruct:


Summary:
The expected amount of memory needed to run this model is about 8.4 GB
The lower and upper bounds are 7.2 - 9.9 GB
If you have 1 GPUs, you will need about 8.4 GB, with bounds of 7.2 - 9.9 GB per GPU


Component Breakdown:
Each GPU will need 7.6 GB to store the model parameters
Each GPU will need 0.0 GB to store the optimizer states
Each GPU will need 0.0 GB to store the gradients
Each GPU will need 0.0 GB to store the intermediate activations
Each GPU will need 0.0 GB to store the outputs
Up to 2.3 GB can be expected as overhead

Decision:
The proposed training setup should work for your hardware.


Estimate

In [ ]:
# Uncomment to run 8B QLoRA training
# Warning: will take significantly longer than 2B

RUN_8B = False  # Set to True to attempt 8B training

if RUN_8B:
    CKPT_DIR_8B = "checkpoints/lora_sft_granite8b"
    os.makedirs(CKPT_DIR_8B, exist_ok=True)

    # Clear GPU memory from previous run
    torch.cuda.empty_cache()
    import gc; gc.collect()

    print("Starting QLoRA training on Granite 8B...")
    start_time = time.time()

    try:
        result_8b = lora_sft(
            model_path=MODEL_8B,
            data_path=os.path.abspath(DATA_PATH),
            ckpt_output_dir=os.path.abspath(CKPT_DIR_8B),
            load_in_4bit=True,
            lora_r=16,
            lora_alpha=32,
            lora_dropout=0.0,
            num_epochs=1,
            effective_batch_size=4,  # Smaller batch for 8B
            max_seq_len=512,
            learning_rate=1e-4,
            lr_scheduler="cosine",
            warmup_steps=10,
            bf16=True,
            nproc_per_node=1,
            logging_steps=25,
            save_steps=500,
        )
        elapsed = time.time() - start_time
        print(f"\n8B training completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")
    except Exception as e:
        elapsed = time.time() - start_time
        print(f"\n8B training failed after {elapsed:.1f}s")
        print(f"Error: {e}")
else:
    print("8B training skipped. Set RUN_8B=True to attempt.")

## 9. Troubleshooting Log

This section documents issues encountered and workarounds attempted.

### Known Issues on DGX Spark

| Issue | Detail | Workaround |
|-------|--------|------------|
| Compute capability 12.1 | PyTorch warns max supported is 12.0 | Usually works anyway — warning only |
| Unsloth wheels | No pre-built wheels for ARM aarch64 | Manual PEFT/TRL fallback (Section 6) |
| xformers | No ARM wheels available | Use PyTorch's built-in SDPA attention |
| CUDA arch | sm_121 not in default TORCH_CUDA_ARCH_LIST | May need `TORCH_CUDA_ARCH_LIST="12.1"` for source builds |

### What Worked
- *Fill in after running*

### What Didn't Work
- *Fill in after running*

### Key Observations
- *Fill in after running*

## 10. Results & Next Steps

### Summary
- **Algorithm:** LoRA + SFT (QLoRA 4-bit)
- **Model:** Granite 2B (primary), Granite 8B (stretch)
- **Hardware:** DGX Spark (GB10, 130.7 GB unified memory)
- **Training time:** *Fill in after running*
- **Peak memory:** *Fill in after running*
- **Status:** *Fill in after running*

### Recommendations
- LoRA/QLoRA is the most accessible training method on DGX Spark
- The massive shared memory means even 8B models can be QLoRA'd comfortably
- ARM + Blackwell ecosystem is still maturing — expect dependency challenges

### Next
- [Notebook 2: Full SFT →](02_sft_spark.ipynb)
- [Notebook 3: OSFT →](03_osft_spark.ipynb)